## 1. Data Skewness
In a perfectly balanced cluster, every executor finishes its task at roughly the same time. When skew occurs:
* **The Symptom:** Most tasks finish in seconds, but one or two tasks take minutes (or hours).
* **The Cause:** A specific **Join Key** or **Group By Key** is over-represented (e.g., a "Null" key or a very popular category like "General Merchandise").
* **The Result:** `OutofMemory (OOM)` errors or underutilization of cluster resources.
---

## 2. Common Solutions in PySpark

### A. Salting (The Most Robust Fix)
Salting involves adding a random integer (a "salt") to the join key to break up large chunks of data into smaller, manageable pieces.

1.  **Modify the skewed table:** Append a random number to the key (e.g., `key_1`, `key_2`).
2.  **Modify the lookup table:** Explode the rows so each original key matches every possible salted version.
3.  **Join:** Perform the join on the new salted keys.

### B. Broadcast Join
If one of the DataFrames is small enough to fit in the memory of an executor, you can bypass the "Shuffle" phase entirely.
* **How it works:** Spark sends a copy of the small DataFrame to every executor.
* **Code:**

```python
from pyspark.sql.functions import broadcast
df_large.join(broadcast(df_small), "key")
```

### C. Adaptive Query Execution (AQE)
Available in Spark 3.0+, AQE can detect skew at runtime and automatically handle it by splitting skewed partitions into smaller ones.
* **Enable it:**

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

### D. Filtering/Handling Nulls
Often, skew is caused by a massive amount of `NULL` values that all get sent to the same partition.
* **Solution:** Filter out nulls before the join if they aren't needed, or fill them with a unique random string so they distribute evenly across the cluster.
---

## 3. Comparison Table

| Solution           | Best Used When...                                     | Complexity |
|:-------------------|:------------------------------------------------------|:-----------|
| **Broadcast Join** | One table is small (< 10MB default, up to a few GBs). | Low        |
| **AQE**            | Using Spark 3.x+; provides a hands-off fix.           | Low        |
| **Salting**        | Both tables are large and one has a heavy key skew.   | High       |
| **Filtering**      | Skew is driven by junk data or Nulls.                 | Medium     |

In [1]:
# importing libs
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, StructField, StructType, IntegerType
from pyspark.sql.functions import col, concat, lit, rand, explode, array, broadcast

In [2]:
# spark configs
spark = SparkSession.Builder().master("local[*]").appName("skewProblem").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "false") # disabling the AQE
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # disable broadcast join to test salting
spark.conf.set("spark.sql.shuffle.partitions", 2) # restricting partition size to 2

In [3]:
# creating skew dataset
skew_data = [("A", i) for i in range(1000000)] + [("B", 1), ("C", 1), ("D", 1)]
skew_data_schema = StructType([
    StructField("key", StringType(), nullable=False),
    StructField("value", IntegerType(), nullable=False)
])
skew_df = spark.createDataFrame(data=skew_data, schema=skew_data_schema)

In [4]:
# lookup df
lookup_data = [("A", "alpha"), ("B", "beta"), ("C", "gamma"), ("D", "delta")]
lookup_df = spark.createDataFrame(lookup_data, ["key", "label"])

In [5]:
# adding salted key in skew df
salted_skew_df = skew_df.withColumn("salted_key", concat(col("key"), lit("_"), (rand() * 11).cast("int")))
salted_skew_df.show(5)

+---+-----+----------+
|key|value|salted_key|
+---+-----+----------+
|  A|    0|       A_4|
|  A|    1|       A_9|
|  A|    2|       A_4|
|  A|    3|       A_8|
|  A|    4|       A_1|
+---+-----+----------+
only showing top 5 rows



In [6]:
# adding salt in lookup df
lookup_df_salt = (lookup_df.withColumn("salt", array([lit(i) for i in range(0, 11)]))
                  .withColumn("exploded", explode(col("salt")))
                  .withColumn("exploded_col", concat(col("key"), lit("_"), col("exploded")))).drop(col("exploded")).drop(col("salt"))
lookup_df_salt.show(5)

+---+-----+------------+
|key|label|exploded_col|
+---+-----+------------+
|  A|alpha|         A_0|
|  A|alpha|         A_1|
|  A|alpha|         A_2|
|  A|alpha|         A_3|
|  A|alpha|         A_4|
+---+-----+------------+
only showing top 5 rows



In [7]:
finalDf = salted_skew_df.join(lookup_df_salt, salted_skew_df.salted_key == lookup_df_salt.exploded_col, "inner")
finalDf.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|alpha|1000000|
|delta|      1|
| beta|      1|
|gamma|      1|
+-----+-------+



## Broadcast Join
A **broadcast join** in PySpark is an optimization technique used to speed up joins when one of the datasets is **small enough to fit in memory**.

#### What is Broadcast Join?
In a normal join, Spark shuffles data across the cluster → **expensive**.
In a broadcast join:
* The **small dataset is sent (broadcasted)** to all worker nodes
* Each worker joins it with its partition of the large dataset
* No shuffle needed → ✅ Faster execution


#### When to Use Broadcast Join
Use it when:
* One table is **very small** (e.g., lookup/dimension table)
* Other table is **large** (fact table)
* Small table can fit in memory on each executor


👉 Typical size guideline:
* Default threshold: **10 MB** (configurable)

Disable broadcast:
```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
```

#### Types of Broadcast Joins
* Broadcast Hash Join (most common)
* Broadcast Nested Loop Join (used when no join condition or non-equi joins)


#### Advantages
- Eliminates shuffle
- Faster joins
- Efficient for star-schema queries
- Reduces network I/O


#### Disadvantages
- Not suitable for large tables
- Can cause **memory issues (OOM)** if broadcast table is too big
- Serialization overhead


#### How Spark Decides Automatically
Spark automatically uses broadcast join if:
* Table size < `spark.sql.autoBroadcastJoinThreshold`
* Join type supports broadcasting


#### Best Practices
* ✔ Always broadcast **dimension tables**
* ✔ Use `.explain()` to verify:
* ✔ Cache small table if reused multiple times
* ✔ Avoid broadcasting borderline large tables


#### Interview Tips
* Broadcast join avoids **shuffle**
* Works best for **small + large** table joins
* Improves performance significantly in **ETL pipelines**
* Controlled via **threshold config**

In [8]:
skew_df.join(broadcast(lookup_df), on=skew_df.key==lookup_df.key, how="inner").groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|alpha|1000000|
|delta|      1|
| beta|      1|
|gamma|      1|
+-----+-------+

